# 13 · Gold: features, labels and device health

**Who this is for:** you, after notebook 12 (Silver).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Run the pipeline `heavden-medallion` first (`databricks bundle run medallion -t dev`), so the Gold tables exist.

**By the end you will be able to:**
- explain the **point-in-time rule**: a feature at hour `t` uses only readings from before `t`
- say why a label is only **final** 12 hours after its hour, and why unfinished hours are left out instead of counted as 0
- rebuild one feature by hand from Silver and check Gold got the same number
- show which diagnoses set each condition flag, and why only six conditions count
- read device health: uptime, outages and stuck sensors

The pipeline file is `databricks/src/pipelines/medallion/gold.py`; the logic is `heavden.pipelines.gold`, tested on local Spark against the pandas reference (`tests/test_gold.py`).

## 1. Setup

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

from heavden.pipelines import gold

dbutils.widgets.text("catalog", "heavden_dev")
CATALOG = dbutils.widgets.get("catalog")
print(repo_root, CATALOG)

## 2. What Gold holds

| Table | One row per | Used by |
|---|---|---|
| `gold.patient_hour_features` | stay and hour on the ward | the model (scoring), the assistant |
| `gold.training_set` | stay and hour whose label is **final** | training and evaluation |
| `gold.device_health_daily` | device and day | the Clinical Ops dashboard, drift investigations |
| `gold.condition_flag_reference` | diagnosis or drug text that sets a flag | auditing the condition flags |

All four are **materialized views**: each refresh recomputes them from Silver. A reading that arrives late, or a corrected record, simply appears in the next refresh; nothing has to be patched by hand.

In [ ]:
for table in [
    "patient_hour_features",
    "training_set",
    "device_health_daily",
    "condition_flag_reference",
]:
    n = spark.table(f"{CATALOG}.gold.{table}").count()
    print(f"{table:28} {n:>8,} rows")

## 3. The data window and the prediction grid

Gold takes its window from the readings themselves: from the first reading's hour to the hour after the last one (`gold.data_window`). Each stay then gets one row per whole hour on the ward: the first is the first full hour after admission, the last is before discharge (or before the end of the data).

In [ ]:
display(gold.data_window(spark.table(f"{CATALOG}.silver.vitals")))

In [ ]:
display(
    spark.sql(f"""
    SELECT e.encounter_id, e.admit_ts, e.discharge_ts,
           min(f.prediction_ts) AS first_hour, max(f.prediction_ts) AS last_hour, count(*) AS hours
    FROM {CATALOG}.gold.patient_hour_features f
    JOIN {CATALOG}.silver.encounters e ON e.encounter_id = f.encounter_id AND e.__END_AT IS NULL
    GROUP BY ALL ORDER BY e.admit_ts LIMIT 10
""")
)

## 4. The point-in-time rule, checked by hand

`heart_rate_mean_1h` at hour `t` is the mean of the readings in `[t - 1h, t)`: the hour **before** `t`, never the hour after. If a feature peeked past `t`, the model would look brilliant in testing and fail on the ward (this is **leakage**).

Pick one patient-hour and rebuild the number from Silver:

In [ ]:
row = spark.sql(f"""
    SELECT encounter_id, prediction_ts, heart_rate_mean_1h, readings_1h
    FROM {CATALOG}.gold.patient_hour_features
    WHERE readings_1h > 0 ORDER BY prediction_ts DESC LIMIT 1
""").first()
print(row)

display(
    spark.sql(f"""
    SELECT avg(heart_rate) AS heart_rate_mean_1h, count(*) AS readings_1h
    FROM {CATALOG}.silver.vitals
    WHERE encounter_id = '{row.encounter_id}'
      AND ts >= TIMESTAMP'{row.prediction_ts}' - INTERVAL 1 HOUR
      AND ts <  TIMESTAMP'{row.prediction_ts}'
""")
)

The two should agree. The 3- and 6-hour means weight each hour by its number of readings, and `missing_1h` / `missing_6h` count expected readings that never came (12 per hour): a monitor that goes quiet is itself a warning sign.

NEWS2 is computed from the 1-hour **medians**, with COPD patients on SpO2 scale 2 and the latest nurse observation for oxygen and consciousness. Same rules as `heavden.ml.news2`:

In [ ]:
display(
    spark.sql(f"""
    SELECT news2_total, count(*) AS hours
    FROM {CATALOG}.gold.patient_hour_features
    GROUP BY 1 ORDER BY 1
""")
)

## 5. Labels from recorded outcomes

`label` = 1 if the patient was escalated (rapid response or ICU) in the 6 hours after `t`. Two delays matter:

1. the **window**: you only know nothing happened once 6 hours have passed;
2. the **recording delay**: escalations are charted about 6 hours after they happen, and only then land in Silver.

So a 0 is final only at `t + 12h`. A recorded escalation settles a 1 straight away. Hours in between are **left out** of `gold.training_set`, not guessed as 0; counting them as 0 would teach the model that the last hours before any escalation were calm.

In [ ]:
display(
    spark.sql(f"""
    SELECT f.n_hours, t.n_labelled, f.n_hours - t.n_labelled AS not_final_yet, t.positives
    FROM (SELECT count(*) AS n_hours FROM {CATALOG}.gold.patient_hour_features) f,
         (SELECT count(*) AS n_labelled, sum(label) AS positives FROM {CATALOG}.gold.training_set) t
""")
)

In [ ]:
display(
    spark.sql(f"""
    SELECT max(prediction_ts) AS last_labelled_hour, max(label_known_at) AS last_label_final_at
    FROM {CATALOG}.gold.training_set
""")
)

## 6. Which diagnoses set the condition flags

The model sees six long-term conditions (COPD, heart failure, diabetes, chronic kidney disease, hypertension, atrial fibrillation) plus beta-blockers. They are exactly the conditions the simulation gives an effect on vital signs; everything else in a patient's history (hundreds of descriptions, from prediabetes to social findings) is left out on purpose. Flags are taken **at admission**: a condition counts if it had started and not stopped by the admission date.

This table shows what each flag actually matched:

In [ ]:
display(spark.table(f"{CATALOG}.gold.condition_flag_reference"))

In a real hospital this list would be clinically reviewed: ischaemic heart disease, anaemia or dementia matter for deterioration, but the synthetic data gives them no effect, so the model can't learn from them (Model Card, known limitations).

## 7. Device health

In [ ]:
display(
    spark.sql(f"""
    SELECT site_id, date, count(*) AS devices, round(avg(uptime_pct), 1) AS mean_uptime_pct,
           sum(battery_outages) AS outages, sum(stuck_minutes) AS stuck_minutes,
           round(avg(mean_spo2), 2) AS mean_spo2
    FROM {CATALOG}.gold.device_health_daily
    GROUP BY ALL ORDER BY date, site_id
""")
)

`uptime_pct` compares messages received with those expected while the device was on a patient (one every 5 minutes). A **stuck** reading is part of a run of 6+ identical heart-rate or blood-pressure values (30 minutes), which those vitals don't do naturally. A site-wide drop in `mean_spo2` can be a faulty sensor batch, not sicker patients.

## 8. Try it yourself

1. Pick a patient with an escalation (`silver.outcomes`). List their `gold.training_set` rows for the 8 hours before it. Which hours have `label = 1`, and why does the hour right after the escalation have none?
2. Find the stay with the most `missing_6h` in one hour. What does `gold.device_health_daily` say about its device that day?
3. Count patients per flag in `gold.condition_flag_reference`, and compare with the share of patients with each flag in `gold.patient_hour_features`. Why can the two differ? (Hint: active at admission.)
4. Change one rule in `heavden.pipelines.gold` locally (e.g. `RECORDING_DELAY_H = 0`) and run `uv run pytest tests/test_gold.py`. Which test catches it?

In [ ]:
# Try it yourself: 1


In [ ]:
# Try it yourself: 2


**Pitfall: labels from outcomes not yet recorded.** A 0 written at `t + 6h` might turn into a 1 when the chart arrives 6 hours later. That's why the label waits 12 hours.

**Pitfall: leakage through windows.** "Last hour" means `[t - 1h, t)`. A window that includes `t` itself lets the model see the moment it's predicting.

**Next:** Phase 2 trains the model on `gold.training_set` with MLflow and registers it in Unity Catalog.